# 11 - Statistical Significance Tests

Bu notebook, aynı iç test örnekleri üzerinde çalışan modelleri **örnek bazında eşleştirerek**
istatistiksel karşılaştırma yapar.

## Üretilen analizler

1. **McNemar testi**
   - İki modelin aynı örneklerde doğru/yanlış davranışlarının farklılaşıp farklılaşmadığını sınar.
   - Çok sınıflı problemde de model tahminleri `doğru / yanlış` durumuna indirgenerek uygulanır.
   - `b` ve `c` discordant örnekler üzerinden hesaplanır.
   - Discordant örnek sayısı küçükse exact binomial McNemar, büyükse continuity-corrected chi-square kullanılır.

2. **Paired bootstrap %95 güven aralığı**
   - Aynı test örnekleri birlikte yeniden örneklenir.
   - Her modelin Macro-F1 değeri ve **iki model arasındaki Macro-F1 farkı** için güven aralığı üretir.
   - Örnek eşleşmesi bozulmaz.

3. **Holm çoklu-karşılaştırma düzeltmesi**
   - Aynı seed içinde birden fazla model çifti test edildiği için McNemar p-değerlerine Holm düzeltmesi uygulanır.

## Beklenen girişler

Önce şu notebookların tamamlanmış olması beklenir:

- `08_finetune_finbert_target_dataset.ipynb`
- `10_multiseed_training.ipynb`

`10` notebooku her yeni seed için `test_predictions.csv` kaydeder. Bu notebook bu dosyaları otomatik keşfeder.

> Not: Eski `seed=42` BERT/DistilBERT/RoBERTa deneylerinde örnek-bazlı prediction CSV kaydedilmemişse
> seed=42 için McNemar testi yapılamaz. Bu durumda seed=123 ve seed=2024 üzerinden paired testler
> çalışır. Notebook eksik dosya için sonuç uydurmaz.


In [1]:
from thesis_utils import PROJECT_ROOT, paths

from pathlib import Path
from itertools import combinations
import json
import math
import re
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import f1_score, accuracy_score

try:
    from scipy.stats import binomtest, chi2
except Exception as exc:
    raise ImportError(
        "Bu notebook scipy gerektirir. Önce `pip install scipy` çalıştır."
    ) from exc

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")


# ============================================================
# 1) AYARLAR
# ============================================================
RANDOM_STATE = 42
N_BOOTSTRAP = 5000
CI_LEVEL = 0.95
ALPHA = 0.05

LABELS_ORDER = ["negative", "neutral", "positive"]

BASE_CHECKPOINT_ROOT = paths.MODEL_CHECKPOINT_ROOT
MULTISEED_ROOT = BASE_CHECKPOINT_ROOT / "multiseed"
FINBERT_08_ROOT = BASE_CHECKPOINT_ROOT / "finbert_target_finetuned_seed42"

OUTPUT_DIR = paths.MODEL_RESULTS_ROOT / "statistical_significance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Checkpoint root :", BASE_CHECKPOINT_ROOT)
print("Multiseed root  :", MULTISEED_ROOT)
print("Output dir      :", OUTPUT_DIR)
print("Bootstrap draws :", N_BOOTSTRAP)


Checkpoint root : D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model
Multiseed root  : D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\multiseed
Output dir      : D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\statistical_significance
Bootstrap draws : 5000


In [2]:
# ============================================================
# 2) PREDICTION DOSYALARINI KEŞFET
# ============================================================

def normalize_model_name(value):
    s = str(value).strip().lower()

    if "distilbert" in s:
        return "DistilBERT-base-uncased"
    if "roberta" in s:
        return "RoBERTa-base"
    if "finbert" in s:
        return "Fine-tuned FinBERT"
    if "bert" in s:
        return "BERT-base-uncased"

    return str(value).strip()


def infer_seed_from_path(path):
    match = re.search(r"seed(\d+)", str(path).lower())
    return int(match.group(1)) if match else None


def infer_model_from_path(path):
    s = str(path).lower()

    if "distilbert" in s:
        return "DistilBERT-base-uncased"
    if "roberta" in s:
        return "RoBERTa-base"
    if "finbert" in s:
        return "Fine-tuned FinBERT"
    if re.search(r"(^|[/\\_])bert([/\\_]|$)", s):
        return "BERT-base-uncased"

    return None


candidate_paths = []

# 10 notebook çıktıları
if MULTISEED_ROOT.exists():
    candidate_paths.extend(MULTISEED_ROOT.rglob("test_predictions.csv"))

# 08 notebook çıktısı
finbert08_pred = FINBERT_08_ROOT / "results" / "test_predictions.csv"
if finbert08_pred.exists():
    candidate_paths.append(finbert08_pred)

# Legacy prediction dosyaları varsa otomatik yakala.
# Sadece checkpoint ağacı içinde test prediction isimlerine bakılır.
if BASE_CHECKPOINT_ROOT.exists():
    for pattern in [
        "*test*predictions*.csv",
        "*test*prediction*.csv",
    ]:
        candidate_paths.extend(BASE_CHECKPOINT_ROOT.rglob(pattern))

# Tekilleştir
candidate_paths = sorted(set(Path(p) for p in candidate_paths))

records = []

required_any_gold = {"gold_label", "label"}
required_any_pred = {"prediction", "predicted_label"}

for path in candidate_paths:
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        try:
            df = pd.read_csv(path)
        except Exception:
            continue

    cols = set(df.columns)

    if not (cols & required_any_gold):
        continue
    if not (cols & required_any_pred):
        continue

    model = (
        normalize_model_name(df["model"].iloc[0])
        if "model" in df.columns and len(df)
        else infer_model_from_path(path)
    )

    seed = (
        int(df["seed"].iloc[0])
        if "seed" in df.columns and len(df) and pd.notna(df["seed"].iloc[0])
        else infer_seed_from_path(path)
    )

    # 08 FinBERT dosyasında seed kolonu yoksa seed=42.
    if seed is None and FINBERT_08_ROOT in path.parents:
        seed = 42

    if model is None or seed is None:
        continue

    records.append({
        "model": model,
        "seed": int(seed),
        "path": path,
        "n_rows": len(df),
    })

discovery_df = pd.DataFrame(records)

if discovery_df.empty:
    raise FileNotFoundError(
        "İstatistiksel test için test_predictions.csv bulunamadı.\n"
        "Önce 08 ve 10 notebooklarını çalıştır."
    )

# Aynı model+seed için birden fazla dosya varsa en açık 'results/test_predictions.csv'
# dosyasına öncelik ver, sonra ilkini al.
discovery_df["priority"] = discovery_df["path"].astype(str).map(
    lambda x: 0 if x.replace("\\", "/").endswith("/results/test_predictions.csv") else 1
)

discovery_df = (
    discovery_df
    .sort_values(["model", "seed", "priority", "path"])
    .drop_duplicates(["model", "seed"], keep="first")
    .drop(columns="priority")
    .reset_index(drop=True)
)

print("Bulunan prediction dosyaları:")
display(discovery_df.assign(path=discovery_df["path"].astype(str)))


Bulunan prediction dosyaları:


,model,seed,path,n_rows
0,BERT-base-uncased,123,D:\serkan.kaymak\financial_sentiment_thesis\fi...,2386
1,BERT-base-uncased,2024,D:\serkan.kaymak\financial_sentiment_thesis\fi...,2386
2,DistilBERT-base-uncased,123,D:\serkan.kaymak\financial_sentiment_thesis\fi...,2386
3,DistilBERT-base-uncased,2024,D:\serkan.kaymak\financial_sentiment_thesis\fi...,2386
4,Fine-tuned FinBERT,42,D:\serkan.kaymak\financial_sentiment_thesis\fi...,2386
5,Fine-tuned FinBERT,123,D:\serkan.kaymak\financial_sentiment_thesis\fi...,2386
6,Fine-tuned FinBERT,2024,D:\serkan.kaymak\financial_sentiment_thesis\fi...,2386
7,RoBERTa-base,123,D:\serkan.kaymak\financial_sentiment_thesis\fi...,2386
8,RoBERTa-base,2024,D:\serkan.kaymak\financial_sentiment_thesis\fi...,2386


In [3]:
# ============================================================
# 3) PREDICTION DOSYALARINI STANDARTLAŞTIR
# ============================================================

def standardize_prediction_df(path, model, seed):
    df = pd.read_csv(path, encoding="utf-8-sig")

    # Gold label
    if "gold_label" in df.columns:
        gold_col = "gold_label"
    elif "label" in df.columns:
        gold_col = "label"
    else:
        raise ValueError(f"Gold label kolonu bulunamadı: {path}")

    # Prediction
    if "prediction" in df.columns:
        pred_col = "prediction"
    elif "predicted_label" in df.columns:
        pred_col = "predicted_label"
    else:
        raise ValueError(f"Prediction kolonu bulunamadı: {path}")

    out = pd.DataFrame()
    out["gold_label"] = df[gold_col].astype(str).str.lower().str.strip()
    out["prediction"] = df[pred_col].astype(str).str.lower().str.strip()

    # En güvenli join anahtarı sample_id.
    if "sample_id" in df.columns:
        out["sample_id"] = df["sample_id"].astype(str)
    else:
        # Legacy dosyada sample_id yoksa satır sırasını kullan.
        # Bu yalnızca aynı test splitinin aynı sırayla kaydedildiği durumda güvenlidir.
        out["sample_id"] = [f"test_{i:06d}" for i in range(len(df))]

    if out["sample_id"].duplicated().any():
        raise ValueError(f"Duplicate sample_id bulundu: {path}")

    invalid_gold = sorted(set(out["gold_label"]) - set(LABELS_ORDER))
    invalid_pred = sorted(set(out["prediction"]) - set(LABELS_ORDER))

    if invalid_gold:
        raise ValueError(f"Geçersiz gold label: {invalid_gold} | {path}")
    if invalid_pred:
        raise ValueError(f"Geçersiz prediction: {invalid_pred} | {path}")

    out["model"] = model
    out["seed"] = int(seed)
    out["correct"] = out["gold_label"] == out["prediction"]

    return out


prediction_frames = {}

for row in discovery_df.itertuples(index=False):
    key = (row.model, int(row.seed))
    prediction_frames[key] = standardize_prediction_df(
        row.path,
        row.model,
        int(row.seed),
    )

print("Standartlaştırılan model/seed kombinasyonları:")
for key, df in prediction_frames.items():
    print(f"{key}: N={len(df)}")


Standartlaştırılan model/seed kombinasyonları:
('BERT-base-uncased', 123): N=2386
('BERT-base-uncased', 2024): N=2386
('DistilBERT-base-uncased', 123): N=2386
('DistilBERT-base-uncased', 2024): N=2386
('Fine-tuned FinBERT', 42): N=2386
('Fine-tuned FinBERT', 123): N=2386
('Fine-tuned FinBERT', 2024): N=2386
('RoBERTa-base', 123): N=2386
('RoBERTa-base', 2024): N=2386


In [4]:
# ============================================================
# 4) EŞLEŞME KONTROLÜ
# ============================================================

def align_pair(df_a, df_b, model_a, model_b, seed):
    a = df_a[["sample_id", "gold_label", "prediction", "correct"]].copy()
    b = df_b[["sample_id", "gold_label", "prediction", "correct"]].copy()

    merged = a.merge(
        b,
        on="sample_id",
        how="inner",
        suffixes=("_a", "_b"),
        validate="one_to_one",
    )

    if len(merged) != len(a) or len(merged) != len(b):
        raise ValueError(
            f"Örnek eşleşmesi eksik: seed={seed}, {model_a} vs {model_b}. "
            f"A={len(a)}, B={len(b)}, ortak={len(merged)}"
        )

    mismatch = merged["gold_label_a"] != merged["gold_label_b"]
    if mismatch.any():
        raise ValueError(
            f"Gold label uyuşmazlığı bulundu: seed={seed}, "
            f"{model_a} vs {model_b}, n={int(mismatch.sum())}"
        )

    merged = merged.rename(columns={"gold_label_a": "gold_label"})
    merged = merged.drop(columns="gold_label_b")

    return merged


available_seeds = sorted({seed for _, seed in prediction_frames})
pair_availability = []

for seed in available_seeds:
    models = sorted([
        model
        for (model, s) in prediction_frames
        if s == seed
    ])

    for model_a, model_b in combinations(models, 2):
        try:
            merged = align_pair(
                prediction_frames[(model_a, seed)],
                prediction_frames[(model_b, seed)],
                model_a,
                model_b,
                seed,
            )
            pair_availability.append({
                "seed": seed,
                "model_a": model_a,
                "model_b": model_b,
                "n_common": len(merged),
                "status": "OK",
            })
        except Exception as exc:
            pair_availability.append({
                "seed": seed,
                "model_a": model_a,
                "model_b": model_b,
                "n_common": np.nan,
                "status": f"ERROR: {exc}",
            })

pair_availability_df = pd.DataFrame(pair_availability)

print("Pair availability:")
display(pair_availability_df)


Pair availability:


,seed,model_a,model_b,n_common,status
0,123,BERT-base-uncased,DistilBERT-base-uncased,2386,OK
1,123,BERT-base-uncased,Fine-tuned FinBERT,2386,OK
2,123,BERT-base-uncased,RoBERTa-base,2386,OK
3,123,DistilBERT-base-uncased,Fine-tuned FinBERT,2386,OK
4,123,DistilBERT-base-uncased,RoBERTa-base,2386,OK
5,123,Fine-tuned FinBERT,RoBERTa-base,2386,OK
6,2024,BERT-base-uncased,DistilBERT-base-uncased,2386,OK
7,2024,BERT-base-uncased,Fine-tuned FinBERT,2386,OK
8,2024,BERT-base-uncased,RoBERTa-base,2386,OK
9,2024,DistilBERT-base-uncased,Fine-tuned FinBERT,2386,OK


In [5]:
# ============================================================
# 5) McNEMAR TESTİ
# ============================================================

def mcnemar_test(correct_a, correct_b):
    correct_a = np.asarray(correct_a, dtype=bool)
    correct_b = np.asarray(correct_b, dtype=bool)

    # b: A doğru, B yanlış
    b = int(np.sum(correct_a & ~correct_b))

    # c: A yanlış, B doğru
    c = int(np.sum(~correct_a & correct_b))

    discordant = b + c

    if discordant == 0:
        return {
            "b_a_correct_b_wrong": b,
            "c_a_wrong_b_correct": c,
            "discordant": discordant,
            "method": "no_discordance",
            "statistic": 0.0,
            "p_value": 1.0,
        }

    if discordant < 25:
        # Exact McNemar = two-sided binomial test with p=0.5.
        result = binomtest(
            k=min(b, c),
            n=discordant,
            p=0.5,
            alternative="two-sided",
        )

        return {
            "b_a_correct_b_wrong": b,
            "c_a_wrong_b_correct": c,
            "discordant": discordant,
            "method": "exact_binomial",
            "statistic": float(min(b, c)),
            "p_value": float(result.pvalue),
        }

    # Continuity-corrected McNemar chi-square
    statistic = (abs(b - c) - 1) ** 2 / discordant
    p_value = chi2.sf(statistic, df=1)

    return {
        "b_a_correct_b_wrong": b,
        "c_a_wrong_b_correct": c,
        "discordant": discordant,
        "method": "chi_square_continuity_corrected",
        "statistic": float(statistic),
        "p_value": float(p_value),
    }


mcnemar_rows = []

for row in pair_availability_df.itertuples(index=False):
    if row.status != "OK":
        continue

    merged = align_pair(
        prediction_frames[(row.model_a, row.seed)],
        prediction_frames[(row.model_b, row.seed)],
        row.model_a,
        row.model_b,
        row.seed,
    )

    result = mcnemar_test(
        merged["correct_a"].to_numpy(),
        merged["correct_b"].to_numpy(),
    )

    acc_a = accuracy_score(
        merged["gold_label"],
        merged["prediction_a"],
    )
    acc_b = accuracy_score(
        merged["gold_label"],
        merged["prediction_b"],
    )

    mcnemar_rows.append({
        "seed": row.seed,
        "model_a": row.model_a,
        "model_b": row.model_b,
        "n": len(merged),
        "accuracy_a": acc_a,
        "accuracy_b": acc_b,
        "accuracy_diff_a_minus_b": acc_a - acc_b,
        **result,
    })

mcnemar_df = pd.DataFrame(mcnemar_rows)

display(mcnemar_df.round(6))


,seed,model_a,model_b,n,accuracy_a,accuracy_b,accuracy_diff_a_minus_b,b_a_correct_b_wrong,c_a_wrong_b_correct,discordant,method,statistic,p_value
0,123,BERT-base-uncased,DistilBERT-base-uncased,2386,0.875943,0.858759,0.017184,112,71,183,chi_square_continuity_corrected,8.743169,0.003108
1,123,BERT-base-uncased,Fine-tuned FinBERT,2386,0.875943,0.877200,-0.001257,86,89,175,chi_square_continuity_corrected,0.022857,0.879829
2,123,BERT-base-uncased,RoBERTa-base,2386,0.875943,0.901090,-0.025147,80,140,220,chi_square_continuity_corrected,15.822727,0.000070
3,123,DistilBERT-base-uncased,Fine-tuned FinBERT,2386,0.858759,0.877200,-0.018441,85,129,214,chi_square_continuity_corrected,8.640187,0.003288
4,123,DistilBERT-base-uncased,RoBERTa-base,2386,0.858759,0.901090,-0.042330,80,181,261,chi_square_continuity_corrected,38.314176,0.000000
5,123,Fine-tuned FinBERT,RoBERTa-base,2386,0.877200,0.901090,-0.023889,81,138,219,chi_square_continuity_corrected,14.319635,0.000154
6,2024,BERT-base-uncased,DistilBERT-base-uncased,2386,0.872590,0.863370,0.009220,96,74,170,chi_square_continuity_corrected,2.594118,0.107261
7,2024,BERT-base-uncased,Fine-tuned FinBERT,2386,0.872590,0.878039,-0.005448,83,96,179,chi_square_continuity_corrected,0.804469,0.369760
8,2024,BERT-base-uncased,RoBERTa-base,2386,0.872590,0.900671,-0.028080,82,149,231,chi_square_continuity_corrected,18.857143,0.000014
9,2024,DistilBERT-base-uncased,Fine-tuned FinBERT,2386,0.863370,0.878039,-0.014669,84,119,203,chi_square_continuity_corrected,5.694581,0.017017


In [6]:
# ============================================================
# 6) HOLM DÜZELTMESİ
# ============================================================

def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)

    if m == 0:
        return np.array([], dtype=float)

    order = np.argsort(p_values)
    sorted_p = p_values[order]

    adjusted_sorted = np.empty(m, dtype=float)
    running_max = 0.0

    for i, p in enumerate(sorted_p):
        adjusted = (m - i) * p
        running_max = max(running_max, adjusted)
        adjusted_sorted[i] = min(running_max, 1.0)

    adjusted = np.empty(m, dtype=float)
    adjusted[order] = adjusted_sorted
    return adjusted


if not mcnemar_df.empty:
    pieces = []

    for seed, group in mcnemar_df.groupby("seed", sort=True):
        g = group.copy()
        g["p_value_holm"] = holm_adjust(g["p_value"].to_numpy())
        g["significant_raw_005"] = g["p_value"] < ALPHA
        g["significant_holm_005"] = g["p_value_holm"] < ALPHA
        pieces.append(g)

    mcnemar_df = pd.concat(pieces, ignore_index=True)

mcnemar_path = OUTPUT_DIR / "mcnemar_pairwise_results.csv"
mcnemar_df.to_csv(mcnemar_path, index=False, encoding="utf-8-sig")

print("McNemar + Holm:")
display(
    mcnemar_df[
        [
            "seed",
            "model_a",
            "model_b",
            "n",
            "b_a_correct_b_wrong",
            "c_a_wrong_b_correct",
            "p_value",
            "p_value_holm",
            "significant_holm_005",
        ]
    ].round(6)
)

print("Saved:", mcnemar_path)


McNemar + Holm:


,seed,model_a,model_b,n,b_a_correct_b_wrong,c_a_wrong_b_correct,p_value,p_value_holm,significant_holm_005
0,123,BERT-base-uncased,DistilBERT-base-uncased,2386,112,71,0.003108,0.009323,True
1,123,BERT-base-uncased,Fine-tuned FinBERT,2386,86,89,0.879829,0.879829,False
2,123,BERT-base-uncased,RoBERTa-base,2386,80,140,0.000070,0.000348,True
3,123,DistilBERT-base-uncased,Fine-tuned FinBERT,2386,85,129,0.003288,0.009323,True
4,123,DistilBERT-base-uncased,RoBERTa-base,2386,80,181,0.000000,0.000000,True
5,123,Fine-tuned FinBERT,RoBERTa-base,2386,81,138,0.000154,0.000617,True
6,2024,BERT-base-uncased,DistilBERT-base-uncased,2386,96,74,0.107261,0.214522,False
7,2024,BERT-base-uncased,Fine-tuned FinBERT,2386,83,96,0.369760,0.369760,False
8,2024,BERT-base-uncased,RoBERTa-base,2386,82,149,0.000014,0.000070,True
9,2024,DistilBERT-base-uncased,Fine-tuned FinBERT,2386,84,119,0.017017,0.051052,False


Saved: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\statistical_significance\mcnemar_pairwise_results.csv


In [7]:
# ============================================================
# 7) PAIRED BOOTSTRAP: Macro-F1 VE FARKI
# ============================================================

def percentile_ci(values, ci_level=0.95):
    values = np.asarray(values, dtype=float)

    alpha = 1.0 - ci_level
    lower = np.quantile(values, alpha / 2)
    upper = np.quantile(values, 1 - alpha / 2)

    return float(lower), float(upper)


def paired_bootstrap_macro_f1(
    merged,
    n_bootstrap=5000,
    random_state=42,
):
    rng = np.random.default_rng(random_state)

    y_true = merged["gold_label"].to_numpy()
    pred_a = merged["prediction_a"].to_numpy()
    pred_b = merged["prediction_b"].to_numpy()

    n = len(merged)

    observed_a = f1_score(
        y_true,
        pred_a,
        labels=LABELS_ORDER,
        average="macro",
        zero_division=0,
    )
    observed_b = f1_score(
        y_true,
        pred_b,
        labels=LABELS_ORDER,
        average="macro",
        zero_division=0,
    )
    observed_diff = observed_a - observed_b

    boot_a = np.empty(n_bootstrap, dtype=float)
    boot_b = np.empty(n_bootstrap, dtype=float)
    boot_diff = np.empty(n_bootstrap, dtype=float)

    for i in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)

        yt = y_true[idx]
        pa = pred_a[idx]
        pb = pred_b[idx]

        f1a = f1_score(
            yt,
            pa,
            labels=LABELS_ORDER,
            average="macro",
            zero_division=0,
        )
        f1b = f1_score(
            yt,
            pb,
            labels=LABELS_ORDER,
            average="macro",
            zero_division=0,
        )

        boot_a[i] = f1a
        boot_b[i] = f1b
        boot_diff[i] = f1a - f1b

    a_low, a_high = percentile_ci(boot_a, CI_LEVEL)
    b_low, b_high = percentile_ci(boot_b, CI_LEVEL)
    d_low, d_high = percentile_ci(boot_diff, CI_LEVEL)

    # İki yönlü bootstrap p benzeri oran:
    # fark dağılımının sıfırın karşı tarafında kalan kütlesi.
    p_boot = 2 * min(
        np.mean(boot_diff <= 0),
        np.mean(boot_diff >= 0),
    )
    p_boot = min(float(p_boot), 1.0)

    return {
        "f1_macro_a": float(observed_a),
        "f1_macro_a_ci_low": a_low,
        "f1_macro_a_ci_high": a_high,
        "f1_macro_b": float(observed_b),
        "f1_macro_b_ci_low": b_low,
        "f1_macro_b_ci_high": b_high,
        "f1_macro_diff_a_minus_b": float(observed_diff),
        "diff_ci_low": d_low,
        "diff_ci_high": d_high,
        "diff_ci_excludes_zero": bool(d_low > 0 or d_high < 0),
        "bootstrap_two_sided_p_approx": p_boot,
    }


bootstrap_rows = []

for row in pair_availability_df.itertuples(index=False):
    if row.status != "OK":
        continue

    merged = align_pair(
        prediction_frames[(row.model_a, row.seed)],
        prediction_frames[(row.model_b, row.seed)],
        row.model_a,
        row.model_b,
        row.seed,
    )

    # Her model/seed çifti için tekrarlanabilir ama farklı bootstrap akışı.
    pair_seed = (
        RANDOM_STATE
        + int(row.seed) * 100
        + sum(ord(ch) for ch in (row.model_a + row.model_b))
    )

    result = paired_bootstrap_macro_f1(
        merged,
        n_bootstrap=N_BOOTSTRAP,
        random_state=pair_seed,
    )

    bootstrap_rows.append({
        "seed": row.seed,
        "model_a": row.model_a,
        "model_b": row.model_b,
        "n": len(merged),
        **result,
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)

bootstrap_path = OUTPUT_DIR / "paired_bootstrap_macro_f1_results.csv"
bootstrap_df.to_csv(
    bootstrap_path,
    index=False,
    encoding="utf-8-sig",
)

print("Paired bootstrap Macro-F1 results:")
display(
    bootstrap_df[
        [
            "seed",
            "model_a",
            "model_b",
            "f1_macro_a",
            "f1_macro_b",
            "f1_macro_diff_a_minus_b",
            "diff_ci_low",
            "diff_ci_high",
            "diff_ci_excludes_zero",
            "bootstrap_two_sided_p_approx",
        ]
    ].round(6)
)

print("Saved:", bootstrap_path)


Paired bootstrap Macro-F1 results:


,seed,model_a,model_b,f1_macro_a,f1_macro_b,f1_macro_diff_a_minus_b,diff_ci_low,diff_ci_high,diff_ci_excludes_zero,bootstrap_two_sided_p_approx
0,123,BERT-base-uncased,DistilBERT-base-uncased,0.842223,0.821273,0.020950,0.006980,0.035292,True,0.0020
1,123,BERT-base-uncased,Fine-tuned FinBERT,0.842223,0.846285,-0.004062,-0.017879,0.009498,False,0.5748
2,123,BERT-base-uncased,RoBERTa-base,0.842223,0.875692,-0.033469,-0.049285,-0.018101,True,0.0000
3,123,DistilBERT-base-uncased,Fine-tuned FinBERT,0.821273,0.846285,-0.025011,-0.040391,-0.009754,True,0.0004
4,123,DistilBERT-base-uncased,RoBERTa-base,0.821273,0.875692,-0.054418,-0.070811,-0.037700,True,0.0000
5,123,Fine-tuned FinBERT,RoBERTa-base,0.846285,0.875692,-0.029407,-0.044567,-0.013791,True,0.0000
6,2024,BERT-base-uncased,DistilBERT-base-uncased,0.838647,0.826368,0.012279,-0.001420,0.026344,False,0.0788
7,2024,BERT-base-uncased,Fine-tuned FinBERT,0.838647,0.846423,-0.007776,-0.021282,0.005963,False,0.2732
8,2024,BERT-base-uncased,RoBERTa-base,0.838647,0.877657,-0.039011,-0.055053,-0.022785,True,0.0000
9,2024,DistilBERT-base-uncased,Fine-tuned FinBERT,0.826368,0.846423,-0.020055,-0.034661,-0.006073,True,0.0068


Saved: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\statistical_significance\paired_bootstrap_macro_f1_results.csv


In [8]:
# ============================================================
# 8) MODEL BAZINDA TEKİL BOOTSTRAP %95 CI
# ============================================================

def bootstrap_single_model_macro_f1(
    df,
    n_bootstrap=5000,
    random_state=42,
):
    rng = np.random.default_rng(random_state)

    y_true = df["gold_label"].to_numpy()
    y_pred = df["prediction"].to_numpy()
    n = len(df)

    observed = f1_score(
        y_true,
        y_pred,
        labels=LABELS_ORDER,
        average="macro",
        zero_division=0,
    )

    boot = np.empty(n_bootstrap, dtype=float)

    for i in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)

        boot[i] = f1_score(
            y_true[idx],
            y_pred[idx],
            labels=LABELS_ORDER,
            average="macro",
            zero_division=0,
        )

    low, high = percentile_ci(boot, CI_LEVEL)

    return observed, low, high


single_ci_rows = []

for (model, seed), df in sorted(
    prediction_frames.items(),
    key=lambda x: (x[0][1], x[0][0]),
):
    model_seed = RANDOM_STATE + seed * 1000 + sum(ord(ch) for ch in model)

    observed, low, high = bootstrap_single_model_macro_f1(
        df,
        n_bootstrap=N_BOOTSTRAP,
        random_state=model_seed,
    )

    single_ci_rows.append({
        "model": model,
        "seed": seed,
        "n": len(df),
        "f1_macro": observed,
        "ci95_low": low,
        "ci95_high": high,
        "ci95_text": f"{observed:.4f} [{low:.4f}, {high:.4f}]",
    })

single_ci_df = pd.DataFrame(single_ci_rows)

single_ci_path = OUTPUT_DIR / "model_macro_f1_bootstrap_ci.csv"
single_ci_df.to_csv(
    single_ci_path,
    index=False,
    encoding="utf-8-sig",
)

display(single_ci_df)
print("Saved:", single_ci_path)


,model,seed,n,f1_macro,ci95_low,ci95_high,ci95_text
0,Fine-tuned FinBERT,42,2386,0.851176,0.833973,0.868089,"0.8512 [0.8340, 0.8681]"
1,BERT-base-uncased,123,2386,0.842223,0.824648,0.858310,"0.8422 [0.8246, 0.8583]"
2,DistilBERT-base-uncased,123,2386,0.821273,0.803145,0.839091,"0.8213 [0.8031, 0.8391]"
3,Fine-tuned FinBERT,123,2386,0.846285,0.828942,0.862910,"0.8463 [0.8289, 0.8629]"
4,RoBERTa-base,123,2386,0.875692,0.859939,0.890871,"0.8757 [0.8599, 0.8909]"
5,BERT-base-uncased,2024,2386,0.838647,0.821434,0.855775,"0.8386 [0.8214, 0.8558]"
6,DistilBERT-base-uncased,2024,2386,0.826368,0.808121,0.843810,"0.8264 [0.8081, 0.8438]"
7,Fine-tuned FinBERT,2024,2386,0.846423,0.829195,0.863097,"0.8464 [0.8292, 0.8631]"
8,RoBERTa-base,2024,2386,0.877657,0.862398,0.892744,"0.8777 [0.8624, 0.8927]"


Saved: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\statistical_significance\model_macro_f1_bootstrap_ci.csv


In [9]:
# ============================================================
# 9) TEZ İÇİN KISA ÖZET TABLOSU
# ============================================================

summary_rows = []

for row in bootstrap_df.itertuples(index=False):
    mc = mcnemar_df[
        (mcnemar_df["seed"] == row.seed)
        & (mcnemar_df["model_a"] == row.model_a)
        & (mcnemar_df["model_b"] == row.model_b)
    ]

    if len(mc) == 1:
        mc_row = mc.iloc[0]
        p_holm = float(mc_row["p_value_holm"])
        mc_sig = bool(mc_row["significant_holm_005"])
    else:
        p_holm = np.nan
        mc_sig = False

    if row.f1_macro_diff_a_minus_b > 0:
        better = row.model_a
    elif row.f1_macro_diff_a_minus_b < 0:
        better = row.model_b
    else:
        better = "Tie"

    summary_rows.append({
        "seed": row.seed,
        "model_a": row.model_a,
        "model_b": row.model_b,
        "better_by_macro_f1": better,
        "macro_f1_diff_a_minus_b": row.f1_macro_diff_a_minus_b,
        "bootstrap_diff_ci_low": row.diff_ci_low,
        "bootstrap_diff_ci_high": row.diff_ci_high,
        "bootstrap_ci_excludes_zero": row.diff_ci_excludes_zero,
        "mcnemar_p_holm": p_holm,
        "mcnemar_significant_holm_005": mc_sig,
        "both_support_difference": (
            bool(row.diff_ci_excludes_zero)
            and bool(mc_sig)
        ),
    })

thesis_summary_df = pd.DataFrame(summary_rows)

summary_path = OUTPUT_DIR / "thesis_statistical_test_summary.csv"
thesis_summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

print("Tez için özet:")
display(thesis_summary_df.round(6))

print("\n" + "#" * 100)
print("STATISTICAL SIGNIFICANCE ANALYSIS FINISHED")
print("#" * 100)
print("Output dir:", OUTPUT_DIR)
print("\nDosyalar:")
print("-", mcnemar_path.name)
print("-", bootstrap_path.name)
print("-", single_ci_path.name)
print("-", summary_path.name)


Tez için özet:


,seed,model_a,model_b,better_by_macro_f1,macro_f1_diff_a_minus_b,bootstrap_diff_ci_low,bootstrap_diff_ci_high,bootstrap_ci_excludes_zero,mcnemar_p_holm,mcnemar_significant_holm_005,both_support_difference
0,123,BERT-base-uncased,DistilBERT-base-uncased,BERT-base-uncased,0.020950,0.006980,0.035292,True,0.009323,True,True
1,123,BERT-base-uncased,Fine-tuned FinBERT,Fine-tuned FinBERT,-0.004062,-0.017879,0.009498,False,0.879829,False,False
2,123,BERT-base-uncased,RoBERTa-base,RoBERTa-base,-0.033469,-0.049285,-0.018101,True,0.000348,True,True
3,123,DistilBERT-base-uncased,Fine-tuned FinBERT,Fine-tuned FinBERT,-0.025011,-0.040391,-0.009754,True,0.009323,True,True
4,123,DistilBERT-base-uncased,RoBERTa-base,RoBERTa-base,-0.054418,-0.070811,-0.037700,True,0.000000,True,True
5,123,Fine-tuned FinBERT,RoBERTa-base,RoBERTa-base,-0.029407,-0.044567,-0.013791,True,0.000617,True,True
6,2024,BERT-base-uncased,DistilBERT-base-uncased,BERT-base-uncased,0.012279,-0.001420,0.026344,False,0.214522,False,False
7,2024,BERT-base-uncased,Fine-tuned FinBERT,Fine-tuned FinBERT,-0.007776,-0.021282,0.005963,False,0.369760,False,False
8,2024,BERT-base-uncased,RoBERTa-base,RoBERTa-base,-0.039011,-0.055053,-0.022785,True,0.000070,True,True
9,2024,DistilBERT-base-uncased,Fine-tuned FinBERT,Fine-tuned FinBERT,-0.020055,-0.034661,-0.006073,True,0.051052,False,False



####################################################################################################
STATISTICAL SIGNIFICANCE ANALYSIS FINISHED
####################################################################################################
Output dir: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\statistical_significance

Dosyalar:
- mcnemar_pairwise_results.csv
- paired_bootstrap_macro_f1_results.csv
- model_macro_f1_bootstrap_ci.csv
- thesis_statistical_test_summary.csv


## Sonuçları nasıl yorumlayacağız?

Bu notebook bittikten sonra tez metnini **çıkan değerlere göre** güncelleyeceğiz; önceden
“istatistiksel olarak anlamlıdır” diye bir sonuç varsaymıyoruz.

Kısa yorum kuralı:

- `mcnemar_significant_holm_005 = True`  
  → iki modelin örnek-bazlı doğru/yanlış davranışları arasındaki fark, Holm düzeltmesi sonrasında anlamlıdır.

- `bootstrap_ci_excludes_zero = True`  
  → Macro-F1 farkının paired-bootstrap %95 güven aralığı sıfırı içermemektedir.

- `both_support_difference = True`  
  → hem McNemar hem paired-bootstrap fark lehine kanıt vermektedir.

### Bana göndermen gereken dosyalar

Notebook tamamlandığında şunları yükle:

- `statistical_significance/thesis_statistical_test_summary.csv`
- `statistical_significance/mcnemar_pairwise_results.csv`
- `statistical_significance/paired_bootstrap_macro_f1_results.csv`
- `statistical_significance/model_macro_f1_bootstrap_ci.csv`

Bunlara göre tezdeki istatistiksel anlamlılık ve güven aralığı kısmını birlikte yazacağız.
